In [4]:
import pandas as pd
import matplotlib.pyplot as plt

# Load raw QuickStats data
df = pd.read_csv("../data/raw/quickstats.csv")

# Filter to exactly what we need
yield_df = df[
    (df['Program'] == 'SURVEY') &
    (df['Data Item'] == 'CORN, GRAIN - YIELD, MEASURED IN BU / ACRE') &
    (df['Geo Level'] == 'STATE') &
    (df['State'].isin(['IOWA', 'COLORADO', 'WISCONSIN', 'MISSOURI', 'NEBRASKA']))
].copy()

# Clean value column
yield_df['Value'] = pd.to_numeric(
    yield_df['Value'].str.replace(',', ''), errors='coerce'
)

# Keep only what we need
yield_df = yield_df[['Year', 'State', 'Value']].dropna()
yield_df.columns = ['year', 'state', 'yield_bu_acre']
yield_df = yield_df.sort_values(['state', 'year']).reset_index(drop=True)

# Save
yield_df.to_csv("../data/processed/quickstats_yield.csv", index=False)
print(f"Saved {len(yield_df)} rows")
print(yield_df)

Saved 495 rows
     year      state  yield_bu_acre
0    2005   COLORADO            148
1    2005   COLORADO            137
2    2005   COLORADO            135
3    2005   COLORADO            135
4    2005   COLORADO            130
..    ...        ...            ...
490  2024  WISCONSIN            174
491  2024  WISCONSIN            183
492  2024  WISCONSIN            181
493  2024  WISCONSIN            182
494  2024  WISCONSIN            182

[495 rows x 3 columns]


In [5]:
yield_df = yield_df.groupby(['year', 'state'])['yield_bu_acre'].mean().reset_index()
yield_df.to_csv("../data/processed/quickstats_yield.csv", index=False)
print(f"{len(yield_df)} rows after aggregation")
print(yield_df)

100 rows after aggregation
    year      state  yield_bu_acre
0   2005   COLORADO          137.0
1   2005       IOWA          170.8
2   2005   MISSOURI          105.2
3   2005   NEBRASKA          157.4
4   2005  WISCONSIN          140.4
..   ...        ...            ...
95  2024   COLORADO          122.2
96  2024       IOWA          211.8
97  2024   MISSOURI          182.4
98  2024   NEBRASKA          193.4
99  2024  WISCONSIN          180.4

[100 rows x 3 columns]


In [ ]:
import requests
import pandas as pd
import time

API_KEY = "YOUR_KEY_HERE"  # paste key here

STATES = {
    "IOWA": "FIPS:19",
    "COLORADO": "FIPS:08",
    "WISCONSIN": "FIPS:55",
    "MISSOURI": "FIPS:29",
    "NEBRASKA": "FIPS:31"
}

MONTHS = ["05", "06", "07", "08", "09", "10"]  # May–Oct growing season

records = []

for state_name, state_fips in STATES.items():
    for year in range(2005, 2025):
        for month in MONTHS:
            url = "https://www.ncei.noaa.gov/cdo-web/api/v2/data"
            params = {
                "datasetid": "GSOM",
                "datatypeid": "TAVG,PRCP",
                "locationid": state_fips,
                "startdate": f"{year}-{month}-01",
                "enddate": f"{year}-{month}-01",
                "units": "standard",
                "limit": 100
            }
            headers = {"token": API_KEY}
            r = requests.get(url, params=params, headers=headers)
            if r.status_code == 200 and "results" in r.json():
                for item in r.json()["results"]:
                    item["state"] = state_name
                    item["year"] = year
                    records.append(item)
            time.sleep(0.2)  # rate limit

weather_df = pd.DataFrame(records)
weather_df.to_csv("../data/raw/noaa_weather.csv", index=False)
print(f"Saved {len(weather_df)} rows")
print(weather_df.head())